In [6]:
import os

from langchain.agents import create_agent
from langchain_community.chat_models.tongyi import ChatTongyi
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

# 加载环境变量 .env 文件
load_dotenv()

chatLLM = ChatTongyi(
    model=os.getenv("ALIYUN_TEXT_MODEL"),
)

In [7]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool


@tool
def search_information(query: str) -> str:
    """提供有关给定主题的事实信息。使用此工具查找诸如"法国首都"或"伦敦的天气？"等短语的答案。
    """
    print(f"\n--- 🛠️ 工具调用：search_information，查询：'{query}' ---")

    # 使用预定义结果字典模拟搜索工具。
    simulated_results = {
        "weather in london": "伦敦目前多云，温度为 15°C。",
        "capital of france": "法国的首都是巴黎。",
        "population of earth": "地球的估计人口约为 80 亿人。",
        "tallest mountain": "珠穆朗玛峰是海拔最高的山峰。",
        "default": f"'{query}' 的模拟搜索结果：未找到特定信息，但该主题似乎很有趣。"
    }

    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"--- 工具结果：{result} ---")
    return result


tools = [search_information]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的助手。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(chatLLM, tools, agent_prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({
    "input": "法国的首都是什么？"
})



> Entering new AgentExecutor chain...

Invoking: `search_information` with `{'query': '法国首都'}`



--- 🛠️ 工具调用：search_information，查询：'法国首都' ---
--- 工具结果：'法国首都' 的模拟搜索结果：未找到特定信息，但该主题似乎很有趣。 ---
'法国首都' 的模拟搜索结果：未找到特定信息，但该主题似乎很有趣。法国的首都是巴黎。

> Finished chain.


{'input': '法国的首都是什么？', 'output': '法国的首都是巴黎。'}